# M5 Quantum Trajectories — Kaggle Runner

Thin launcher notebook for scripts in the
[m5-quantum-trajectories](https://github.com/billpage/m5-quantum-trajectories) repository.

**Usage:**
1. Run **Cell 1 (Setup)** once per session — clones the repo and installs CuPy.
2. Edit the `SCRIPT` and `ARGS` variables in **Cell 2 (Run)** and execute.
3. PNG outputs land in `/kaggle/working/` and appear in the notebook output panel.

After pushing changes to GitHub, re-run Cell 1 to `git pull` the latest code.

In [ ]:
# ── Cell 1: Setup (run once per session) ──────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/billpage/m5-quantum-trajectories.git"
REPO_DIR = "/kaggle/working/m5"
SRC_DIR  = os.path.join(REPO_DIR, "src")

# Output directory — all scripts read M5_OUTPUT via m5.utils
os.environ["M5_OUTPUT"] = "/kaggle/working"

# Clone or pull
if os.path.exists(REPO_DIR):
    print("Repo exists — pulling latest …")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)
else:
    print("Cloning repo …")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Show latest commit
subprocess.run(["git", "log", "--oneline", "-3"], cwd=REPO_DIR)

# Install CuPy (quiet; no-op if already present)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Quick GPU check
try:
    import cupy as cp
    print(f"\nGPU ready: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
except Exception as e:
    print(f"\nNo GPU — will fall back to NumPy: {e}")

print(f"\nM5_OUTPUT = {os.environ['M5_OUTPUT']}")
print(f"Source directory: {SRC_DIR}")
print("Scripts available:", sorted(f for f in os.listdir(SRC_DIR) if f.endswith('.py')))

In [ ]:
# ── Cell 2: Run a script ──────────────────────────────────────────
#
# Edit SCRIPT and ARGS, then run this cell.
#
SCRIPT  = "m5_gridless_opt.py"
ARGS    = ["--Np", "500", "--Nt", "200"]
TIMEOUT = 600  # seconds

# ── streaming execution ───────────────────────────────────────────
import subprocess, sys, os, time

script_path = os.path.join(SRC_DIR, SCRIPT)
assert os.path.exists(script_path), f"Not found: {script_path}"

cmd = [sys.executable, "-u", script_path] + ARGS
print(f">>> {' '.join(cmd)}\n")

t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,           # line-buffered
    cwd=SRC_DIR          # so `from m5.utils import ...` works
)

try:
    for line in proc.stdout:
        print(line, end="")
    proc.wait(timeout=TIMEOUT)
except subprocess.TimeoutExpired:
    proc.kill()
    print(f"\n*** TIMEOUT after {TIMEOUT}s ***")

elapsed = time.time() - t0
print(f"\nExit code: {proc.returncode}  ({elapsed:.1f}s)")

In [ ]:
# ── Cell 3: Display PNG outputs ──────────────────────────────────
from IPython.display import display, Image
import glob, os

out = os.environ.get("M5_OUTPUT", "/kaggle/working")
pngs = sorted(glob.glob(os.path.join(out, "*.png")), key=os.path.getmtime)
if not pngs:
    print(f"No PNG files found in {out}")
else:
    for p in pngs:
        print(p)
        display(Image(filename=p))

In [ ]:
# ── Cell 4 (optional): Clean up PNGs before next run ─────────────
import glob, os
out = os.environ.get("M5_OUTPUT", "/kaggle/working")
for p in glob.glob(os.path.join(out, "*.png")):
    os.remove(p)
    print(f"Removed {p}")